# Dataset

In [1]:
!pip install datasets transformers torch

In [2]:
from datasets import load_dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

# Load dataset (IMDb example)
dataset = load_dataset("imdb")

# Split into train/test
train_data = dataset["train"].shuffle(seed=42).select(range(2000))  # subset for demo
test_data = dataset["test"].shuffle(seed=42).select(range(500))

# Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_fn(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

train_data = train_data.map(tokenize_fn, batched=True)
test_data = test_data.map(tokenize_fn, batched=True)

# Set format for Trainer
train_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

# Model and Training (https://arxiv.org/abs/1910.01108)

In [6]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer, IntervalStrategy

# Model
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# TrainingArgs
training_args = TrainingArguments(
    output_dir="./results-full-ft",
    eval_strategy=IntervalStrategy.EPOCH, # Changed from evaluation_strategy to eval_strategy
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_steps=1000,
    logging_steps=50,
    report_to=[]   #wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data
)

# Train (full fine-tuning of DistilBERT)
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.385745,0.539909
2,0.267280,0.596639


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.36013351440429686, metrics={'train_runtime': 62.8346, 'train_samples_per_second': 63.659, 'train_steps_per_second': 7.957, 'total_flos': 132467398656000.0, 'train_loss': 0.36013351440429686, 'epoch': 2.0})

In [7]:
import torch
from transformers import pipeline
import random

class ModelComparison:
    def __init__(self, base_model_name, fine_tuned_model_path, tokenizer):
        self.base_model = DistilBertForSequenceClassification.from_pretrained(base_model_name, num_labels=2)
        self.fine_tuned_model = DistilBertForSequenceClassification.from_pretrained(fine_tuned_model_path)
        self.tokenizer = tokenizer

    def predict_single(self, text, model):
        inputs = self.tokenizer(text, truncation=True, padding=True, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
            prediction = torch.argmax(probabilities, dim=-1)
        return {
            "prediction": "Positive" if prediction.item() == 1 else "Negative",
            "confidence": probabilities[0][prediction.item()].item()
        }

    def compare_predictions(self, texts):
        results = []
        for text in texts:
            base_pred = self.predict_single(text, self.base_model)
            tuned_pred = self.predict_single(text, self.fine_tuned_model)

            results.append({
                "text": text,
                "base_model": base_pred,
                "fine_tuned": tuned_pred
            })
        return results

# Example usage:
def display_comparison():
    # Save the fine-tuned model first
    trainer.save_model("./fine-tuned-distilbert")

    # Initialize comparison
    comparison = ModelComparison(
        "distilbert-base-uncased",
        "./fine-tuned-distilbert",
        tokenizer
    )

    # Example reviews to test
    test_reviews = [
        "This movie was absolutely fantastic! The acting was superb and the story kept me engaged throughout.",
        "Terrible waste of time. The plot made no sense and the characters were poorly developed.",
        "While not perfect, the film had some interesting moments and decent performances.",
        "I'm still not sure how I feel about this one. It had its ups and downs.",
        "The special effects were amazing but the dialogue was pretty cheesy."
    ]

    # Get and display results
    results = comparison.compare_predictions(test_reviews)

    print("\nModel Prediction Comparison:\n")
    print("-" * 80)

    for result in results:
        print(f"\nReview: {result['text']}")
        print("\nBase Model:")
        print(f"Prediction: {result['base_model']['prediction']}")
        print(f"Confidence: {result['base_model']['confidence']:.2%}")
        print("\nFine-tuned Model:")
        print(f"Prediction: {result['fine_tuned']['prediction']}")
        print(f"Confidence: {result['fine_tuned']['confidence']:.2%}")
        print("-" * 80)

# Run the comparison
if __name__ == "__main__":
    display_comparison()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Model Prediction Comparison:

--------------------------------------------------------------------------------

Review: This movie was absolutely fantastic! The acting was superb and the story kept me engaged throughout.

Base Model:
Prediction: Negative
Confidence: 56.61%

Fine-tuned Model:
Prediction: Positive
Confidence: 98.69%
--------------------------------------------------------------------------------

Review: Terrible waste of time. The plot made no sense and the characters were poorly developed.

Base Model:
Prediction: Negative
Confidence: 54.80%

Fine-tuned Model:
Prediction: Negative
Confidence: 99.29%
--------------------------------------------------------------------------------

Review: While not perfect, the film had some interesting moments and decent performances.

Base Model:
Prediction: Negative
Confidence: 54.55%

Fine-tuned Model:
Prediction: Positive
Confidence: 98.28%
--------------------------------------------------------------------------------

Review: I